In [112]:
#all imports
import pandas as pd
import sqlite3
from sklearn import impute , preprocessing
import ydata_profiling
import numpy as np

In [113]:
#Data Understanding & Loading
#All file import

#csv
df_csv = pd.read_csv('users.csv')

#json 
df_json = pd.read_json('sales.json')

#sql
conn = sqlite3.connect('inventory.db')
df_sql = pd.read_sql("SELECT * FROM products", conn)

#merge 
df= pd.merge(df_csv, df_json, on='user_id', how='outer')
df = pd.merge(df, df_sql, on='product_id', how='outer')

df_clean = df.copy()


In [114]:
#Display top 5 records and data info summary
print("First 5 Rows of the DataFrame:")
print(df.head(), "\n")

print("DataFrame Info:")
print(df.info,"\n")

#Identify data types, missing values, and inconsistent records.
print("Data Types:")
print(df.dtypes, "\n")

print("Missing Values:")
print(df.isna().sum(), "\n")

print("Data Description:")
print(df.describe(), "\n")

First 5 Rows of the DataFrame:
  user_id             name  age  gender       city registration_date  \
0   U0003     Aarohi Gupta   37   Other     Indore        2022-02-02   
1   U0010    Diya Kulkarni   36  Female      Delhi        2024-01-18   
2   U0012     Aarohi Singh   28   Other  Ghaziabad        2024-06-25   
3   U0031      Aarohi Khan   27   Other    Lucknow        2022-04-25   
4   U0032  Kabir Mukherjee   46    Male      Thane        2022-01-04   

  transaction_id product_id  amount payment_type       date product_name  \
0        T000631       P001   24.37       Wallet 2024-12-04  Product_001   
1        T000379       P001   22.48          UPI 2023-08-08  Product_001   
2        T000679       P001   54.06   Debit Card 2023-04-05  Product_001   
3        T000667       P001   64.81       Wallet 2023-10-09  Product_001   
4        T000720       P001   61.34  Credit Card 2024-03-31  Product_001   

  category   price  stock  
0  Grocery  264.89    371  
1  Grocery  264.89    3

In [115]:
#Data Cleaning

#Handle missing numerical data using SimpleImputer (mean strategy).
df_clean['age'] = impute.SimpleImputer(strategy='mean').fit_transform(df_clean[['age']])

#Handle missing categorical data using most frequent imputation.
df_clean['city'] = impute.SimpleImputer(strategy='most_frequent').fit_transform(df_clean[['city']]).flatten()

#Fix invalid or inconsistent entries (e.g., wrong date formats, negative prices, etc.)
df_clean['price'] = df_clean['price'].apply(lambda x: abs(x) if x < 0 else x)
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')


In [116]:
#Outlier Handling

#Detect and remove outliers using Z-score and IQR method.


#Z-Score method
m = df_clean['amount'].mean()
print("Mean of amount Before Outlier Removal:", m)

s = df_clean['amount'].std()
print("Standard Deviation of amount Before Outlier Removal:", s, "\n")

#Z-score calculation
z = (df_clean['amount']-m)/s

#removing outliers
df_clean_amount = df_clean[(z>=-3) & (z<=3)]

m = df_clean_amount['amount'].mean()
print("Mean of amount After Outlier Removal (z-score):", m)

s = df_clean_amount['amount'].std()
print("Standard Deviation of amount After Outlier Removal (z-score):", s, "\n")

print('shape before outlier removal (Z-score):', df_clean.shape[0])
print('shape after outlier removal (Z-score):', df_clean_amount.shape[0], "\n")

#IQR method
q1 = np.quantile(df_clean['amount'], 0.25)
q3 = np.quantile(df_clean['amount'], 0.75)

IQR = q3 - q1

lower_bound = q1 - 1.5*IQR
upper_bound = q3 + 1.5*IQR

#removing outliers
df_clean_amount_iqr = df_clean[(df_clean['amount']>=lower_bound) & (df_clean['amount']<=upper_bound)]

print('shape before outlier removal (IQR):', df_clean.shape[0])
print('shape after outlier removal (IQR):', df_clean_amount_iqr.shape[0])


Mean of amount Before Outlier Removal: 67.59904
Standard Deviation of amount Before Outlier Removal: 45.411374511386605 

Mean of amount After Outlier Removal (z-score): 64.63044670050762
Standard Deviation of amount After Outlier Removal (z-score): 37.404385190877996 

shape before outlier removal (Z-score): 1000
shape after outlier removal (Z-score): 985 

shape before outlier removal (IQR): 1000
shape after outlier removal (IQR): 947


### Compare both techniques and decide which is more suitable for this dataset.

- The Z-score method did not significantly reduce the dataset size, indicating that extreme outliers beyond ±3 standard deviations were not present. However, the IQR method removed several data points, showing its effectiveness in detecting moderate outliers. Therefore, the IQR method is more suitable for this dataset as it is more robust for non-normal distributions.

In [117]:
#Winsorization
df_win = df_clean.copy()

p5 = np.percentile(df['amount'], 5)
p95 = np.percentile(df['amount'], 95)

df_win['amount'] = df['amount'].clip(lower=p5, upper=p95)

print("Mean of Main Data: ", df['amount'].mean())
print("Mean of Winsorized Data: ", df_win['amount'].mean())

Mean of Main Data:  67.59904
Mean of Winsorized Data:  65.451415


In [118]:
#Data Transformation

#Convert date columns into separate day, month, year features
df_clean['day'] = df_clean['date'].dt.day
df_clean['month'] = df_clean['date'].dt.month
df_clean['year'] = df_clean['date'].dt.year

#Encode categorical variables using:

#Label Encoding for binary columns.
df_clean['payment_type_label'] = preprocessing.LabelEncoder().fit_transform(df_clean[['payment_type']])

#One-Hot Encoding for nominal columns.
one_hot_encoding_gender= preprocessing.OneHotEncoder().fit_transform(df_clean[['gender']]).toarray()

#Ordinal Encoding for ordered categorical variables.
df_clean['cateory_ordinal'] = preprocessing.OrdinalEncoder().fit_transform(df_clean[['category']])

#Apply binning (e.g., segment customers into spending groups: Low, Medium, High).
df_clean['price_category'] = pd.cut(df_clean['price'], bins=[0, 500, 1000, 1500, 5000], labels=['Low', 'Medium', 'High', 'Very High'])

#Apply log and square root transformations to normalize skewed data.
df_clean['amount_log'] = preprocessing.FunctionTransformer(np.log1p).fit_transform(df_clean[['amount']])
df_clean['amount_sqrt'] = preprocessing.FunctionTransformer(np.sqrt).fit_transform(df_clean[['amount']])

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [119]:
#Feature Scaling

#Use StandardScaler and MinMaxScaler to scale numerical features
df_clean['amount_scaler'] = preprocessing.StandardScaler().fit_transform(df_clean[['amount']])
df_clean['amount_minmax'] = preprocessing.MinMaxScaler().fit_transform(df_clean[['amount']])

print("Summary of amount after standard scaling: \n", df_clean['amount_scaler'].describe(),"\n")
print("Summary of amount after min-max scaling: \n", df_clean['amount_minmax'].describe(),"\n")

Summary of amount after standard scaling: 
 count    1.000000e+03
mean    -3.552714e-17
std      1.000500e+00
min     -1.317268e+00
25%     -6.577422e-01
50%     -2.469568e-01
75%      3.378808e-01
max      1.064916e+01
Name: amount_scaler, dtype: float64 

Summary of amount after min-max scaling: 
 count    1000.000000
mean        0.110080
std         0.083609
min         0.000000
25%         0.055115
50%         0.089443
75%         0.138316
max         1.000000
Name: amount_minmax, dtype: float64 



In [120]:
#Feature Construction

#Average monthly spend per customer.
df_clean['avg_monthly_spend'] = df_clean.groupby('user_id')['amount'].transform('mean')

#Frequency of purchase.
df_clean['purchase_frequency'] = df_clean.groupby('user_id')['amount'].transform('count')

#Days since last purchase.
df_clean['days_since_last_purchase'] = (pd.to_datetime('today') - df_clean.groupby('user_id')['date'].transform('max')).dt.days

#Category-wise total expenditure.
df_clean['category_total_expenditure'] = df_clean.groupby('category')['amount'].transform('sum')


In [130]:
#Number of records before and after cleaning 

print("Records Before Cleaning:", df.shape[0])
print("Records After Cleaning:", df_clean.shape[0],"\n")

#Number of features created.
features_before = df.shape[1]
features_after = df_clean.shape[1]
features_created = features_after - features_before
print("Features Before:", features_before)
print("Features After:", features_after)
print("New Features Created:", features_created)

#Missing value summary (before vs. after).
missing_before = df.isnull().sum()
missing_after = df_clean.isnull().sum()
missing_summary = pd.DataFrame({'Before Cleaning': missing_before, 'After Cleaning': missing_after})
print(missing_summary, "\n")

#Outlier count (before vs. after).
outliers_before = df[(z < -3) | (z > 3)].shape[0]
outliers_after = df_clean_amount[(z < -3) | (z > 3)].shape[0]
print("Outliers Before Cleaning (Z-score):", outliers_before)
print("Outliers After Cleaning (Z-score):", outliers_after)

Records Before Cleaning: 1000
Records After Cleaning: 1000 

Features Before: 15
Features After: 29
New Features Created: 14
                            Before Cleaning  After Cleaning
age                                     0.0               0
amount                                  0.0               0
amount_log                              NaN               0
amount_minmax                           NaN               0
amount_scaler                           NaN               0
amount_sqrt                             NaN               0
avg_monthly_spend                       NaN               0
category                                0.0               0
category_total_expenditure              NaN               0
cateory_ordinal                         NaN               0
city                                    0.0               0
date                                    0.0               0
day                                     NaN               0
days_since_last_purchase           

C:\Users\Admin\AppData\Local\Temp\ipykernel_26092\1755657334.py:22: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  outliers_after = df_clean_amount[(z < -3) | (z > 3)].shape[0]


In [133]:
# Data Profiling Report
report = ydata_profiling.ProfileReport(df_clean, title="Customer Purchase Behavior Analysis Report", explorative=True)
report.to_file("customer_purchase_behavior_analysis_report.html")

Summarize dataset:  86%|████████▌ | 31/36 [00:00<00:00, 61.78it/s, Calculate auto correlation]                   c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 2 1 2 9 0 0 3 2 8 1 1 5 2 9 3 7 3 2 7 3 5 3 2 0 2 4 7 6 8 0 3 7 1 7 7 1
 2 8 8 4 8 8 1 7 8 7 6 6 2 6 7 4 2 7 0 8 9 2 5 1 3 4 0 2 7 8 8 0 9 2 6 9 0
 2 9 8 1 4 8 0 0 2 2 7 0 5 6 5 0 7 2 0 0 6 4 8 1 3 3 5 8 9 2 9 4 8 5 4 8 0
 6 3 9 9 0 1 9 0 8 1 2 2 4 8 2 0 5 6 5 0 0 9 5 6 3 9 5 0 1 2 6 0 7 6 2 0 6
 9 5 9 0 3 4 4 0 0 4 7 6 3 6 0 7 0 2 5 7 5 8 8 4 2 8 2 5 2 3 2 2 8 3 1 5 7
 9 6 7 4 9 6 2 4 8 0 4 8 5 4 3 0 1 6 3 8 1 2 0 0 8 4 9 3 5 1 4 4 3 7 6 8 1
 9 0 1 6 9 0 7 4 8 7 1 0 3 2 2 0 4 9 2 6 9 7 9 2 6 6 0 2 6 0 9 9 4 7 6 0 5
 0 8 2 2 6 9 4 5 9 8 2 0 5 6 3 9 8 9 9 0 0 3 0 4 7 8 6 3 2 1 7 9 8 4 0 4 7
 1 6 2 7 3 2 6 6 7 3 0 2 6 7 5 8 3 0 4 1 9

In [134]:
#Save cleaned data as final_cleaned_dataset.csv.

df_clean.to_csv('final_cleaned_dataset.csv', index=False)

In [ ]:
# Data Profiling Report
report = ydata_profiling.ProfileReport(df_clean, title="Customer Purchase Behavior Analysis Report", explorative=True)
report.to_file("customer_purchase_behavior_analysis_report.html")

Summarize dataset:  86%|████████▌ | 31/36 [00:00<00:00, 61.78it/s, Calculate auto correlation]                   c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 2 1 2 9 0 0 3 2 8 1 1 5 2 9 3 7 3 2 7 3 5 3 2 0 2 4 7 6 8 0 3 7 1 7 7 1
 2 8 8 4 8 8 1 7 8 7 6 6 2 6 7 4 2 7 0 8 9 2 5 1 3 4 0 2 7 8 8 0 9 2 6 9 0
 2 9 8 1 4 8 0 0 2 2 7 0 5 6 5 0 7 2 0 0 6 4 8 1 3 3 5 8 9 2 9 4 8 5 4 8 0
 6 3 9 9 0 1 9 0 8 1 2 2 4 8 2 0 5 6 5 0 0 9 5 6 3 9 5 0 1 2 6 0 7 6 2 0 6
 9 5 9 0 3 4 4 0 0 4 7 6 3 6 0 7 0 2 5 7 5 8 8 4 2 8 2 5 2 3 2 2 8 3 1 5 7
 9 6 7 4 9 6 2 4 8 0 4 8 5 4 3 0 1 6 3 8 1 2 0 0 8 4 9 3 5 1 4 4 3 7 6 8 1
 9 0 1 6 9 0 7 4 8 7 1 0 3 2 2 0 4 9 2 6 9 7 9 2 6 6 0 2 6 0 9 9 4 7 6 0 5
 0 8 2 2 6 9 4 5 9 8 2 0 5 6 3 9 8 9 9 0 0 3 0 4 7 8 6 3 2 1 7 9 8 4 0 4 7
 1 6 2 7 3 2 6 6 7 3 0 2 6 7 5 8 3 0 4 1 9